In [ ]:
#Simple
#Focus: Clean and standardize phone numbers, join phone types, deduplicate.
#Uses (only): Person.PersonPhone, Person.PhoneNumberType
#Creates: silver_person_phone parquet file

In [29]:
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.conf.set("spark.sql.legacy.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.int96RebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInWrite", "CORRECTED")

base = "abfss://fabmigation1@synpasetofabric.dfs.core.windows.net/bronze/adventureworks/Person"

tables = ["PersonPhone","PhoneNumberType"]

dfs = {t: spark.read.parquet(f"{base}.{t}.parquet/") for t in tables}

dfPersonPhone = dfs["PersonPhone"]; dfPhoneNumberType = dfs["PhoneNumberType"]
display(dfs["PersonPhone"])
# display(dfs["Person"].limit(10))

In [30]:
# --- Imports ---
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
silver_base_path = "abfss://fabmigation1@synpasetofabric.dfs.core.windows.net/silver/adventureworks/silver_person_phone/"
source_system = "AdventureWorks"

In [31]:
# Normalize phone numbers to a quasi E.164 (assume default country '1' for NANPA where applicable)
def clean_digits(col):
    return F.regexp_replace(F.col(col), r"[^0-9]", "")

In [32]:
pp = (
    dfPersonPhone
    .withColumn("phone_digits", clean_digits("PhoneNumber"))
    .withColumn("digits_len", F.length("phone_digits"))
    .withColumn(
        "country_code",
        F.when((F.col("digits_len") == 11) & F.col("phone_digits").startswith("1"), F.lit("1"))
         .when(F.col("digits_len") == 10, F.lit("1"))  # assume US if 10 digits
         .otherwise(F.lit(None))
    )
    .withColumn(
        "national_number",
        F.when(F.col("digits_len") == 11, F.substring("phone_digits", 2, 10))
         .when(F.col("digits_len") == 10, F.col("phone_digits"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "phone_e164",
        F.when(F.col("country_code").isNotNull() & F.col("national_number").isNotNull(),
               F.concat(F.lit("+"), F.col("country_code"), F.col("national_number")))
         .otherwise(F.col("PhoneNumber"))
    )
)
display(pp.limit(100))

In [33]:
ppt = dfPhoneNumberType.select(
    "PhoneNumberTypeID",
    F.col("Name").alias("phone_type_name")
)
display(ppt.limit(100))

In [34]:
# Join type & dedupe on (BusinessEntityID, phone_e164)
joined = (
    pp.join(ppt, "PhoneNumberTypeID", "left")
      .withColumn("hash_nk", F.sha2(F.concat_ws("||",
                                                F.col("BusinessEntityID").cast("string"),
                                                F.coalesce(F.col("phone_e164"), F.lit(""))), 256))
)
display(joined.limit(100))

In [35]:
w = Window.partitionBy("hash_nk").orderBy(F.col("ModifiedDate").desc_nulls_last())

In [36]:
silver_df = (
    joined
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "digits_len", "phone_digits")
    .withColumn("person_phone_sk", F.sha2(F.concat_ws("||", F.col("hash_nk"), F.col("PhoneNumberTypeID").cast("string")), 256))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_source", F.lit("Person.PersonPhone"))
    .withColumn("source_system", F.lit(source_system))
)
display(silver_df.limit(100))

In [37]:
silver_df = silver_df.select(
    "person_phone_sk", "BusinessEntityID", "PhoneNumberTypeID", "phone_type_name",
    "PhoneNumber", "phone_e164", "ModifiedDate", "record_source", "source_system", "ingestion_timestamp"
)
display(silver_df.limit(100))

In [54]:
silver_df.write.parquet(silver_base_path)